# Bolverse server on Google Colab

Runs Bolverse's AI server on Colab's free T4 GPU and publishes it at your ngrok address for the web and mobile apps.

**Before the first run**

1. **GPU:** *Runtime → Change runtime type → T4 GPU*.
2. **Project:** upload `bolverse.zip` to Google Drive, in a folder named `bolverse`.
3. **ngrok:** in the Secrets panel (key icon, left), add `NGROK_AUTHTOKEN` and `NGROK_DOMAIN` and switch on *Notebook access*.

**Each session:** *Runtime → Run all*, keep this tab open, and wait for cell 8 to say everything is ready. With the default Light setup the first session downloads about 9 GB and is ready in 10 to 12 minutes.

**Test accounts** (code `123456`, test mode only): +91 90000 00001 to 00007, +1 202 555 0143, +34 612 345 678, +971 50 123 4567. **Admin dashboard:** your address + `/dashboard` (owner@bolverse.test / Owner@12345).

This is for development and testing. Colab sessions end when idle or after a few hours.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU found. Choose Runtime > Change runtime type > T4 GPU, then run this again."

In [ ]:
#@title 2. Settings
#@markdown **Setup.** *Light* is quick and fits Colab's free machine comfortably: one natural voice per language, and Whisper for all speech. *Full* adds separate male and female AI voices and the Indian speech model, but takes twice as long to start and can run out of memory.
SETUP = "Light (recommended)"  #@param ["Light (recommended)", "Full"]
#@markdown **Accounts:** login with mobile number, daily free limit and Premium. Untick for open access while testing.
ACCOUNTS_ENABLED = True  #@param {type:"boolean"}
#@markdown Translation helper model. 3B gives better reading guides; 1.5B is faster and smaller.
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"  #@param ["Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"]
#@markdown **Role.** *Everything* runs the whole app here (for testing). *AI worker* runs only the AI models here, for the main server on your Oracle VM (docs/deploy-oracle-cloud.md).
ROLE = "Everything (testing)"  #@param ["Everything (testing)", "AI worker for my Oracle server"]
#@markdown Free translations per person in total, guests included (Premium is unlimited).
FREE_TRANSLATIONS = 50  #@param {type:"integer"}
DRIVE_FOLDER = "bolverse"  #@param {type:"string"}

FULL = SETUP.startswith("Full")
AI_WORKER = ROLE.startswith("AI worker")
VOICES = FULL                 # separate male and female AI voices (voice worker)
INDIAN_SPEECH = FULL          # AI4Bharat Indian speech model
ASR_MODEL = "openai/whisper-large-v3-turbo"
print("Setup:", "Full: male and female voices, Indian speech model" if FULL else "Light: one voice per language, Whisper for all speech")


In [ ]:
#@title 3. Connect Google Drive and unpack the project
import os, glob, shutil, zipfile
from google.colab import drive

# Fall back to the default settings if cell 2 wasn't run.
for _name, _value in dict(SETUP="Light (recommended)", ACCOUNTS_ENABLED=True, VOICES=False, INDIAN_SPEECH=False,
                          LLM_MODEL="Qwen/Qwen2.5-3B-Instruct", ASR_MODEL="openai/whisper-large-v3-turbo",
                          FREE_TRANSLATIONS=50, DRIVE_FOLDER="bolverse").items():
    globals().setdefault(_name, _value)

drive.mount('/content/drive')
DRIVE = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(f'{DRIVE}/data', exist_ok=True)

zips = sorted(glob.glob(f'{DRIVE}/*.zip'), key=os.path.getmtime)
assert zips, f'No zip found. Upload bolverse.zip to Google Drive > My Drive > {DRIVE_FOLDER}, then run this cell again.'
shutil.rmtree('/content/app', ignore_errors=True)
zipfile.ZipFile(zips[-1]).extractall('/content/app')
found = glob.glob('/content/app/**/backend/app/main.py', recursive=True)
assert found, 'That zip has no backend/app/main.py inside. Upload the Bolverse project zip.'
BACKEND = os.path.dirname(os.path.dirname(found[0]))
print('Using', os.path.basename(zips[-1]), '->', BACKEND)

In [ ]:
#@title 4. Install the server (about 3 minutes)
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -r {BACKEND}/requirements.txt pyngrok bitsandbytes torchaudio
print('Server installed.')

In [ ]:
#@title 5. Install the voice worker (Full setup only; skipped in Light)
# Its own Python 3.12 environment: Kokoro doesn't support Colab's Python 3.13 yet, and Parler-TTS needs an older transformers.
if VOICES:
    !pip install -q uv
    !uv venv -q --python 3.12 /content/voice-env
    !uv pip install -q --python /content/voice-env/bin/python -r {BACKEND}/voice_worker/requirements.txt
    print('Voice worker installed.')
else:
    print('Skipped (Light setup): one voice per language, no voice worker needed.')

In [ ]:
#@title 6. Keys
import json, secrets
from google.colab import userdata

def secret(name):
    try:
        return (userdata.get(name) or '').strip()
    except Exception:
        return ''

#@markdown Optional: paste your ngrok token here to use it instead of the Secret (don't share the notebook if you do).
NGROK_TOKEN_HERE = ""  #@param {type:"string"}
NGROK_AUTHTOKEN = NGROK_TOKEN_HERE.strip() or secret('NGROK_AUTHTOKEN')
print('Using the ngrok token from', 'this cell.' if NGROK_TOKEN_HERE.strip() else 'Secrets (NGROK_AUTHTOKEN).')
# Accept a pasted "ngrok config add-authtoken <token>" command, quotes or stray spaces.
NGROK_AUTHTOKEN = NGROK_AUTHTOKEN.replace('ngrok config add-authtoken', '').strip().strip('"\'').split()[-1] if NGROK_AUTHTOKEN.strip() else ''
if NGROK_AUTHTOKEN and len(NGROK_AUTHTOKEN) < 30:
    print('Warning: this ngrok token looks too short. Copy it again from dashboard.ngrok.com > Your Authtoken.')
NGROK_DOMAIN = secret('NGROK_DOMAIN').replace('https://', '').replace('http://', '').strip('/')
assert NGROK_AUTHTOKEN, 'Add NGROK_AUTHTOKEN in the Secrets panel (key icon on the left) and turn on Notebook access.'

keys_file = f'{DRIVE}/data/keys.json'  # kept on Drive so logins and the API key survive new sessions
keys = json.load(open(keys_file)) if os.path.exists(keys_file) else {}
keys.setdefault('secret_key', secrets.token_hex(32))
keys.setdefault('api_key', secrets.token_urlsafe(24))
keys.setdefault('ai_secret', secrets.token_urlsafe(32))  # shared with the Oracle server in AI worker mode
json.dump(keys, open(keys_file, 'w'))
print(f'ngrok token: {len(NGROK_AUTHTOKEN)} characters, {NGROK_AUTHTOKEN[:3]}...{NGROK_AUTHTOKEN[-3:]}')
print('Keys ready.' + ('' if NGROK_DOMAIN else ' No NGROK_DOMAIN set: the address will change every session.'))

In [ ]:
#@title 7. Start everything and open the public address
import subprocess, sys, time, threading, sqlite3, urllib.request
from pyngrok import ngrok

# Fall back to the default settings if cell 2 wasn't run.
for _name, _value in dict(SETUP="Light (recommended)", ACCOUNTS_ENABLED=True, VOICES=False, INDIAN_SPEECH=False,
                          LLM_MODEL="Qwen/Qwen2.5-3B-Instruct", ASR_MODEL="openai/whisper-large-v3-turbo",
                          FREE_TRANSLATIONS=50, DRIVE_FOLDER="bolverse", AI_WORKER=False).items():
    globals().setdefault(_name, _value)

LOCAL_DB = '/content/bolverse.db'        # SQLite on local disk (fast, safe locking)
DRIVE_DB = f'{DRIVE}/data/bolverse.db'   # copied to Drive every 5 minutes
if os.path.exists(DRIVE_DB) and not os.path.exists(LOCAL_DB):
    shutil.copy(DRIVE_DB, LOCAL_DB)

def backup_db():
    if not os.path.exists(LOCAL_DB):
        return
    src, dst = sqlite3.connect(LOCAL_DB), sqlite3.connect(DRIVE_DB + '.tmp')
    src.backup(dst)
    dst.close(); src.close()
    os.replace(DRIVE_DB + '.tmp', DRIVE_DB)

def wait_for(url, proc, seconds=120):
    for _ in range(seconds // 2):
        try:
            urllib.request.urlopen(url, timeout=2); return True
        except Exception:
            if proc.poll() is not None: return False
            time.sleep(2)
    return False

for name in ('server', 'voice_worker'):
    if name in globals() and globals()[name].poll() is None:
        globals()[name].terminate(); globals()[name].wait()

# Voice worker (male and female voices)
voice_ok = False
if VOICES and os.path.exists('/content/voice-env/bin/python'):
    voice_worker = subprocess.Popen(
        ['/content/voice-env/bin/python', '-m', 'uvicorn', 'voice_worker.worker:app', '--host', '127.0.0.1', '--port', '8001'],
        cwd=BACKEND, stdout=open('/content/voice-worker.log', 'w'), stderr=subprocess.STDOUT)
    voice_ok = wait_for('http://127.0.0.1:8001/health', voice_worker)
    print('Voice worker:', 'running' if voice_ok else 'did not start (see /content/voice-worker.log); using fallback voices')

ngrok.kill()
ngrok.set_auth_token(NGROK_AUTHTOKEN)  # writes ngrok's config file (the most reliable way)
def open_tunnel():
    """Newer ngrok agents call the fixed address 'url'; older ones call it 'domain' (or 'hostname')."""
    if not NGROK_DOMAIN:
        return ngrok.connect(8000, 'http')
    domain = NGROK_DOMAIN.replace('https://', '').replace('http://', '').strip('/')
    errors = []
    for option in ({'url': 'https://' + domain}, {'domain': domain}, {'hostname': domain}):
        try:
            return ngrok.connect(8000, 'http', **option)
        except Exception as exc:
            errors.append(str(exc))
            if 'ERR_NGROK_4018' in str(exc) or 'authentication failed' in str(exc):
                raise RuntimeError('ngrok rejected the token in NGROK_AUTHTOKEN. Copy a fresh one from dashboard.ngrok.com > '
                                   'Your Authtoken (the token only, not the command), paste it into Secrets, then rerun cells 6 and 7.') from None
            ngrok.kill()
    raise RuntimeError(f'ngrok could not open {domain}. Check that this domain is listed under dashboard.ngrok.com > '
                       f'Domains for the same account as the token. ngrok said: {errors[-1][:300]}')

tunnel = open_tunnel()
PUBLIC_URL = tunnel.public_url.replace('http://', 'https://')

env = dict(
    os.environ,
    BOLVERSE_DEVICE='cuda',
    BOLVERSE_ASR_MODEL=ASR_MODEL,
    BOLVERSE_ASR_ENGINE='faster',          # faster-whisper: quicker, skips silence and noise; falls back automatically
    BOLVERSE_ASR_BEAM='5',
    BOLVERSE_INDIC_ASR='true' if INDIAN_SPEECH else 'false',
    BOLVERSE_LLM_MODEL=LLM_MODEL,
    BOLVERSE_LLM_4BIT='true' if '7B' in LLM_MODEL else 'false',
    BOLVERSE_TRANSLATION_BACKEND='nllb',
    BOLVERSE_NLLB_MODEL='facebook/nllb-200-distilled-1.3B',   # clearly better translations than the 600M model; still fits the T4
    BOLVERSE_TTS_ENABLED='true',
    BOLVERSE_VOICE_WORKER_URL='http://127.0.0.1:8001' if voice_ok else '',
    BOLVERSE_PRELOAD='true',
    BOLVERSE_DATABASE_URL=f'sqlite:///{LOCAL_DB}',
    BOLVERSE_DATA_DIR=f'{DRIVE}/data',
    BOLVERSE_SECRET_KEY=keys['secret_key'],
    BOLVERSE_API_KEY='',                 # no API key: every feature already needs a mobile-number login
    BOLVERSE_PUBLIC_BASE_URL=PUBLIC_URL,
    BOLVERSE_CORS_ORIGINS='*',
    APP_ENV='development',                # test accounts, codes shown in the app, simulated payments
    BOLVERSE_SEED_TEST_USERS='true',
    BOLVERSE_SMS_PROVIDER='console',
    BOLVERSE_PAYMENT_PROVIDER_INR='dev',
    BOLVERSE_PAYMENT_PROVIDER_USD='dev',
    BOLVERSE_ACCOUNTS_ENABLED='true' if ACCOUNTS_ENABLED else 'false',
    BOLVERSE_FREE_TRANSLATIONS=str(FREE_TRANSLATIONS),   # in total, guests included; then the paywall
    BOLVERSE_FREE_TRANSLATIONS_PER_DAY='0',
    BOLVERSE_LEARNING_ENABLED="false",
    HF_HUB_DISABLE_PROGRESS_BARS='1',
    BOLVERSE_AI_SECRET=keys['ai_secret'],
)
app_module = 'app.ai_server:app' if AI_WORKER else 'app.main:app'
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', app_module, '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND, env=env, stdout=open('/content/bolverse-server.log', 'w'), stderr=subprocess.STDOUT)
if not wait_for('http://127.0.0.1:8000/' + ('ping' if AI_WORKER else 'health'), server):
    raise RuntimeError('The server stopped while starting. Run the Logs cell below to see why.')
if AI_WORKER:
    print('\nAI worker is running (models load in the background; run cell 8 to watch).')
    print('On your Oracle server, put these two lines in /etc/bolverse/bolverse.env and restart bolverse:')
    print(f'    BOLVERSE_AI_URL={PUBLIC_URL}')
    print(f"    BOLVERSE_AI_SECRET={keys['ai_secret']}")
    print('Use a fixed NGROK_DOMAIN so this address stays the same every session.')

def backup_loop():
    while server.poll() is None:
        time.sleep(300)
        try:
            backup_db()
        except Exception as exc:
            print('Database backup failed:', exc)
threading.Thread(target=backup_loop, daemon=True).start()

if not AI_WORKER:
  print('\nServer is running. Accounts are', 'ON.' if ACCOUNTS_ENABLED else 'OFF (open access).')
  print('Server address:', PUBLIC_URL)
  print('The apps already use this address (DEFAULT_SERVER in apps/web/src/config.ts and apps/mobile/src/config.ts).')
  print('If it differs, change it there.')
  if ACCOUNTS_ENABLED:
      print('\nTest accounts (code 123456): +91 90000 00001 to 00007, +1 202 555 0143, +34 612 345 678, +971 50 123 4567')
      print('Admin dashboard:', PUBLIC_URL + '/dashboard', '  (owner@bolverse.test / Owner@12345)')
  print('\nModels are loading. Run cell 8 to watch.')

In [ ]:
#@title 8. Wait for the models (Light: about 9 GB, 10 to 12 min; Full: about 17 GB)
import json
started = time.time()
while True:
    if AI_WORKER:
        req = urllib.request.Request('http://127.0.0.1:8000/ai/status', headers={'X-AI-Secret': keys['ai_secret']})
        health = json.load(urllib.request.urlopen(req))
    else:
        health = json.load(urllib.request.urlopen('http://127.0.0.1:8000/health'))
    status = '   '.join(f"{name.replace('_', ' ')}: {'ready' if ok else 'loading'}" for name, ok in health['loaded'].items())
    print(f'\r{int(time.time() - started):>4}s   {status}', end='')
    if all(health['loaded'].values()):
        print('\n\nAll models ready. Open the app and start talking.')
        print('Speech engine:', health.get('ai', {}).get('speech_engine') or health.get('speech_engine', 'whisper'))
        break
    if server.poll() is not None:
        print('\n\nThe server stopped. Run the Logs cell to see why.')
        break
    time.sleep(10)

In [ ]:
#@title 9. Test through the public address (logs in as a test user and asks one question)
import urllib.parse

def call(path, body=None, form=None, token=None):
    headers = {'X-API-Key': keys['api_key'], 'X-Device-Id': 'colab-test-device', 'ngrok-skip-browser-warning': '1'}
    if token:
        headers['Authorization'] = 'Bearer ' + token
    data = None
    if body is not None:
        data = json.dumps(body).encode(); headers['Content-Type'] = 'application/json'
    if form is not None:
        data = urllib.parse.urlencode(form).encode(); headers['Content-Type'] = 'application/x-www-form-urlencoded'
    req = urllib.request.Request(PUBLIC_URL + path, data=data, headers=headers)
    return json.load(urllib.request.urlopen(req, timeout=240))

print('Health:', call('/health')['status'])
token = None
if ACCOUNTS_ENABLED:
    call('/auth/otp', {'phone': '+919000000002'})
    token = call('/auth/verify', {'phone': '+919000000002', 'code': '123456'})['token']
answer = call('/ask', form={'source': 'hi-IN', 'target': 'es-ES', 'text': 'main thak gaya hoon'}, token=token)
print('You typed:         ', answer['question'])
print('Understood as:     ', answer['source_text'])
print('In Spanish:        ', answer['translation'])
print('Read it in Hindi:  ', answer['native_reading'])
print('In English letters:', answer['english_reading'])
print('Voices for Hindi:  ', call('/voices/hi-IN', token=token))


## Connect the apps

- **Web:** in WSL, `npm run web` from the project folder, then open http://localhost:5173. The server address is built into the app (`apps/web/src/config.ts`).
- **Phone:** set `EXPO_PUBLIC_API_URL` in `apps/mobile/.env`, run `npx expo start --tunnel`, and scan the QR code with Expo Go.

In [ ]:
#@title Logs
!tail -n 40 /content/bolverse-server.log
!echo '--- voice worker ---' && tail -n 15 /content/voice-worker.log 2>/dev/null || true

In [ ]:
#@title Save the database to Drive now
backup_db()
print('Saved to', DRIVE_DB)

In [ ]:
#@title Stop everything
backup_db()
server.terminate()
if 'voice_worker' in globals():
    voice_worker.terminate()
ngrok.kill()
print('Stopped. The database was saved to Drive.')